<a href="https://colab.research.google.com/github/gokadagaya/Data_prep_Assignment/blob/main/used_cars_data_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
!pip install kagglehub

In [30]:
import kagglehub

path = kagglehub.dataset_download("manishkr1754/cardekho-used-car-data")
print("Path:", path)

Using Colab cache for faster access to the 'cardekho-used-car-data' dataset.
Path: /kaggle/input/cardekho-used-car-data


In [31]:
import os
print(os.listdir(path))

['cardekho_dataset.csv']


In [32]:
import pandas as pd

df = pd.read_csv(path + "/cardekho_dataset.csv")

Step 1: Inspect the data

In [33]:
print(df.info())
print(df.shape)
print(df.isnull().sum()*100/len(df))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   car_name           15411 non-null  object 
 2   brand              15411 non-null  object 
 3   model              15411 non-null  object 
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  object 
 7   fuel_type          15411 non-null  object 
 8   transmission_type  15411 non-null  object 
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-null  float64
 12  seats              15411 non-null  int64  
 13  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(6), object(6)
memory usage: 1.6+ MB
None
(15411, 14)
Unnamed: 0           0.0
car_name           

In [34]:
df=df.dropna(subset=['selling_price'])
df

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15406,19537,Hyundai i10,Hyundai,i10,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5,250000
15407,19540,Maruti Ertiga,Maruti,Ertiga,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7,925000
15408,19541,Skoda Rapid,Skoda,Rapid,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5,425000
15409,19542,Mahindra XUV500,Mahindra,XUV500,5,3800000,Dealer,Diesel,Manual,16.00,2179,140.00,7,1225000


In [35]:
for col in ['mileage','engine','max_power']:
  df[col]=df[col].astype(str).str.replace(r'[^0-9]','',regex=True)
  df[col]=pd.to_numeric(df[col],errors='coerce')
  df[col].fillna(df[col].median(),inplace=True)

/tmp/ipykernel_5901/4245415766.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(),inplace=True)
/tmp/ipykernel_5901/4245415766.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try usi

In [36]:
df=df[(df['selling_price']!=999999999) & (df['selling_price']>=10000)]


In [37]:
df=df.drop_duplicates()

In [38]:
print(df.shape)

(15411, 14)


Task 2

In [39]:
df['transmission_type'] = df['transmission_type'].map({
    'Manual': 0,
    'Automatic': 1
})

In [40]:
df = pd.get_dummies(df, columns=['fuel_type', 'seller_type'], drop_first=True)

In [41]:
print(df.columns.tolist())

['Unnamed: 0', 'car_name', 'brand', 'model', 'vehicle_age', 'km_driven', 'transmission_type', 'mileage', 'engine', 'max_power', 'seats', 'selling_price', 'fuel_type_Diesel', 'fuel_type_Electric', 'fuel_type_LPG', 'fuel_type_Petrol', 'seller_type_Individual', 'seller_type_Trustmark Dealer']


Why is drop_first=True used?

drop_first=True is used to avoid multicollinearity (dummy variable trap).
When we create dummy variables, one category can be predicted from others, so we drop one to prevent redundancy.

What does a row of all zeros represent?

A row with all zeros in the dummy columns represents the base (dropped) category.

For example:
If fuel_type_Petrol and fuel_type_Diesel exist, then

0,0 → represents the dropped category (e.g., CNG)


In [42]:
X = df.drop('selling_price', axis=1)
y = df['selling_price']

In [43]:
X = X.select_dtypes(include=['number'])

In [44]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [45]:
y_pred = [y_train.mean()] * len(y_test)

In [46]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, y_pred)
print(f"Baseline MAE: ₹{round(mae)}")

Baseline MAE: ₹468748
